In [1]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

import lineax as lx

from qd_solve import *
from qd_solve.operator import *
from qd_solve.spaces.pseudospectral import *
from qd_solve.spaces.finite_difference import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

In [2]:
x0 = -10
xf = 10
num_steps = 1000
num_modes = 250

space = PseudoSpectral(x0, xf, num_steps, num_modes)
T = -0.5 * PseudoSpectralLaplacian()

In [3]:
key = jax.random.key(0)
y_vals = jax.random.normal(key, shape=(num_steps,), dtype=jnp.array(1j).dtype)
y = space.from_values(y_vals)
b = T(y)

%time y_sol = jax.block_until_ready(T.solve(b, scale=1.0, shift=0.0))
jnp.linalg.norm((y_sol - y).coeffs) / jnp.linalg.norm(y.coeffs)

CPU times: user 22.8 ms, sys: 0 ns, total: 22.8 ms
Wall time: 41.2 ms


Array(nan, dtype=float64)

In [4]:
func = lambda y: y - 0.1 * T(y)
lx_op = lx.FunctionLinearOperator(func, input_structure=jax.eval_shape(lambda: y))
%time sol = lx.linear_solve(lx_op, y)

CPU times: user 176 ms, sys: 17.3 ms, total: 193 ms
Wall time: 184 ms


In [5]:
jnp.linalg.norm((func(sol.value) - y).coeffs)

Array(1.50134389e-16, dtype=float64)

In [9]:
space = FiniteDifference(x0, xf, num_steps)
L = FiniteDifferenceLaplacian()
b = space.from_values(y_vals) 
y = L.solve(b, scale=-0.1, shift=1.0)

jnp.linalg.norm(((y - 0.1 * L(y)) - b).values)

Array(1.89416354e-13, dtype=float64)